# VELLUM on Mistral 7B, including the one you built

Benchmarks Mistral 7B v0.3 on identical weights and prompts three ways: Hugging Face Transformers, vLLM, and **your from-scratch engine** (`vellum/mistral`, built in `01_build_mistral.ipynb`), once per rung of its optimization ladder (`naive → lastlogit → nosync → gqa → sdpa → chunked → graph → awq → triton`), with AWQ checked against WikiText-2 perplexity. Then vLLM's AWQ and prefix caching under load on the same model.

Needs Colab Pro with an **L4** (Runtime → Change runtime type). 7B in fp16 is 14.5 GB of weights, which a 16 GB T4 cannot hold with a KV cache. Without Colab Pro, run the same suite on Google Cloud with `scripts/gcp_run.sh`. It downloads ~33 GB (HF shards, Mistral's consolidated checkpoint for your engine, vLLM's AWQ build) and takes roughly 2–2.5 hours, so keep the tab open.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
smi = !nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits
mib = int(smi[0])
assert mib >= 20_000, f"Mistral 7B fp16 needs an L4 (24 GB) or bigger; this GPU has {mib / 1024:.0f} GiB"

In [ ]:
REPO_URL = "https://github.com/pr4deepkumar/vellum.git"
!git clone -q $REPO_URL vellum
%cd vellum

In [ ]:
!pip install -q uv
!uv pip install --system vllm
!uv pip install --system -e ".[mistral]"

In [ ]:
import os

# scripts/run_all.sh defaults to the full v2 run on Mistral 7B: HF and vLLM, every rung of the
# from-scratch engine (AWQ included), and vLLM's AWQ and prefix caching under load.
# USD/hr, used only for $/1M tokens. ~0.71 = GCP g2-standard-4 on-demand; the L4 is only sold
# bundled with its VM, so this includes vCPU and RAM.
os.environ["GPU_COST"] = "0.71"
try:  # optional: the Mistral repos are public; an HF_TOKEN Colab secret only lifts Hub rate limits
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
!bash scripts/run_all.sh


In [ ]:
import glob
from IPython.display import Image, Markdown, display
out = sorted(glob.glob('results/*/report.md'))[-1].rsplit('/', 1)[0]
for png in sorted(glob.glob(f'{out}/*.png')):
    display(Image(png))
display(Markdown(open(f'{out}/report.md').read()))

In [ ]:
from google.colab import files
files.download(out + '.tar.gz')